In [ ]:
from pathlib import Path
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# 0. Path configuration
# ============================================================

def find_project_root():
    """
    Locate the project root from either a Python script or a notebook.

    Expected project structure:
    Project_root/
    ├─ Code/
    │  └─ classifier/
    ├─ Data/
    │  └─ ASIGranite.xls
    └─ Result/
    """
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists():
            return path

    if start_dir.name.lower() == "classifier":
        return start_dir.parents[1]

    if start_dir.name.lower() == "code":
        return start_dir.parent

    return start_dir


PROJECT_ROOT = find_project_root()

BASE_DIR = (
    PROJECT_ROOT
    / "Result"
    / "08_Fixed_FiveFold_Multimodel_Performance_Comparison"
)

OUT_DIR = BASE_DIR / "00_All_Classifier_Summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_XLSX = OUT_DIR / "all_models_fixed5fold_summary.xlsx"


# ============================================================
# 1. Global settings
# ============================================================

MODEL_ORDER = [
    "ExtraTrees",
    "GBDT",
    "KNN",
    "LogisticRegression",
    "MLP",
    "RandomForest",
    "SVM"
]

METHOD_ORDER = ["global_mean", "knn"]

FEATURE_SET_ORDER = [
    "Classical_only",
    "Stable_novel_ratios",
    "Classical_plus_stable_novel",
    "Stable_feature_set"
]

CLASS_ORDER = ["A", "S", "I"]

METRIC_COLS = [
    "Accuracy",
    "Balanced_accuracy",
    "Macro_precision",
    "Macro_recall",
    "Macro_F1",
    "Weighted_precision",
    "Weighted_recall",
    "Weighted_F1"
]


MODEL_ORDER_PLOT = [
    "KNN",
    "LogisticRegression",
    "SVM",
    "RandomForest",
    "ExtraTrees",
    "GBDT",
    "MLP"
]

MODEL_LABEL_MAP_PLOT = {
    "KNN": "KNN",
    "LogisticRegression": "LR",
    "SVM": "SVM",
    "RandomForest": "RF",
    "ExtraTrees": "ET",
    "GBDT": "GBDT",
    "MLP": "MLP"
}

FEATURE_SET_ORDER_PLOT = [
    "Classical_only",
    "Stable_novel_ratios",
    "Classical_plus_stable_novel",
    "Stable_feature_set"
]

FEATURE_LABEL_MAP_PLOT = {
    "Classical_only": "Classical only",
    "Stable_novel_ratios": "Stable novel ratios",
    "Classical_plus_stable_novel": "Classical + novel",
    "Stable_feature_set": "Stable feature set"
}


# ============================================================
# 2. Utility functions
# ============================================================

def find_model_result_file(model_dir):
    """
    Find the main fixed-five-fold result workbook for one classifier.

    Priority is given to files ending with '_fixed5fold_results.xlsx'
    to avoid accidentally reading additional metric files.
    """
    files = glob.glob(str(model_dir / "*.xlsx"))

    result_files = [
        Path(file_path)
        for file_path in files
        if file_path.endswith("_fixed5fold_results.xlsx")
    ]

    if result_files:
        return result_files[0]

    result_files = [
        Path(file_path)
        for file_path in files
        if "fixed5fold_results" in Path(file_path).name
    ]

    if result_files:
        return result_files[0]

    return None


def standardize_feature_set_name(value):
    """
    Standardize feature-set names across different script versions.
    """
    s = str(value)

    mapping = {
        "Stable_features_for_paper": "Stable_feature_set",
        "stable_features_for_paper": "Stable_feature_set",
        "Stable features for paper": "Stable_feature_set",
        "Stable_features": "Stable_feature_set"
    }

    return mapping.get(s, s)


def standardize_common_columns(df):
    """
    Standardize common column names across different output versions.
    """
    df = df.copy()

    rename_map = {}

    if "Method" in df.columns:
        rename_map["Method"] = "Imputation_method"

    if "Outer_fold" in df.columns:
        rename_map["Outer_fold"] = "Fold"

    if "index" in df.columns:
        df = df.drop(columns=["index"])

    df = df.rename(columns=rename_map)

    if "Feature_set" in df.columns:
        df["Feature_set"] = df["Feature_set"].apply(standardize_feature_set_name)

    return df


def to_numeric_metrics(df):
    """
    Convert metric columns to numeric values.
    """
    df = df.copy()

    for col in METRIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "N_features" in df.columns:
        df["N_features"] = pd.to_numeric(df["N_features"], errors="coerce")

    if "Fold" in df.columns:
        df["Fold"] = pd.to_numeric(df["Fold"], errors="coerce")

    return df


def sort_by_order(df):
    """
    Sort by predefined model, imputation, feature-set, and fold order.
    """
    df = df.copy()

    if "Model" in df.columns:
        df["Model"] = pd.Categorical(
            df["Model"],
            categories=MODEL_ORDER,
            ordered=True
        )

    if "Imputation_method" in df.columns:
        df["Imputation_method"] = pd.Categorical(
            df["Imputation_method"],
            categories=METHOD_ORDER,
            ordered=True
        )

    if "Feature_set" in df.columns:
        df["Feature_set"] = pd.Categorical(
            df["Feature_set"],
            categories=FEATURE_SET_ORDER,
            ordered=True
        )

    sort_cols = [
        col for col in ["Model", "Imputation_method", "Feature_set", "Fold"]
        if col in df.columns
    ]

    if sort_cols:
        df = df.sort_values(sort_cols)

    for col in ["Model", "Imputation_method", "Feature_set"]:
        if col in df.columns:
            df[col] = df[col].astype(str)

    return df.reset_index(drop=True)


def flatten_columns(df):
    """
    Flatten MultiIndex columns generated by groupby aggregation.
    """
    df = df.copy()

    df.columns = [
        "_".join([str(item) for item in col if str(item) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in df.columns
    ]

    return df


def mean_std_text(mean_value, std_value, digits=3):
    """
    Format mean and standard deviation as text.
    """
    if pd.isna(mean_value):
        return ""

    if pd.isna(std_value):
        return f"{mean_value:.{digits}f}"

    return f"{mean_value:.{digits}f} +/- {std_value:.{digits}f}"


def make_mean_std_table(summary_df, metric):
    """
    Build a mean +/- SD table for one metric, grouped by imputation workflow.
    """
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"

    df = summary_df.copy()

    if mean_col not in df.columns:
        return pd.DataFrame()

    df[f"{metric}_mean_std"] = df.apply(
        lambda row: mean_std_text(row[mean_col], row.get(std_col, np.nan)),
        axis=1
    )

    table = df.pivot_table(
        index=["Model", "Feature_set"],
        columns="Imputation_method",
        values=f"{metric}_mean_std",
        aggfunc="first"
    ).reset_index()

    return table


def make_overall_mean_std_table(overall_summary_df, metric):
    """
    Build an overall mean +/- SD table after combining both imputation workflows.
    """
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"

    df = overall_summary_df.copy()

    if mean_col not in df.columns:
        return pd.DataFrame()

    df[f"{metric}_mean_std"] = df.apply(
        lambda row: mean_std_text(row[mean_col], row.get(std_col, np.nan)),
        axis=1
    )

    table = df.pivot_table(
        index="Model",
        columns="Feature_set",
        values=f"{metric}_mean_std",
        aggfunc="first"
    ).reset_index()

    ordered_cols = ["Model"] + [
        col for col in FEATURE_SET_ORDER
        if col in table.columns
    ]

    table = table[ordered_cols]

    return table


def make_pivot_by_method(summary_df, metric):
    """
    Build a pivot table with one row per Model | Imputation_method
    and one column per feature set.
    """
    metric_col = f"{metric}_mean"

    if metric_col not in summary_df.columns:
        return pd.DataFrame()

    df = summary_df.copy()
    df["Model_method"] = (
        df["Model"].astype(str)
        + " | "
        + df["Imputation_method"].astype(str)
    )

    pivot = df.pivot_table(
        index="Model_method",
        columns="Feature_set",
        values=metric_col,
        aggfunc="first"
    )

    pivot = pivot.reindex(
        columns=[col for col in FEATURE_SET_ORDER if col in pivot.columns]
    )

    return pivot


def make_overall_pivot(overall_summary_df, metric):
    """
    Build an overall pivot table with models as rows and feature sets as columns.
    """
    metric_col = f"{metric}_mean"

    if metric_col not in overall_summary_df.columns:
        return pd.DataFrame()

    pivot = overall_summary_df.pivot_table(
        index="Model",
        columns="Feature_set",
        values=metric_col,
        aggfunc="first"
    )

    pivot = pivot.reindex(
        index=[model for model in MODEL_ORDER if model in pivot.index]
    )

    pivot = pivot.reindex(
        columns=[col for col in FEATURE_SET_ORDER if col in pivot.columns]
    )

    return pivot


def save_heatmap(pivot_df, out_png, title, value_label="Value"):
    """
    Save a heatmap using matplotlib only.
    """
    if pivot_df.empty:
        print("Skipped empty figure:", title)
        return

    plot_df = pivot_df.copy()

    row_labels = plot_df.index.astype(str).tolist()
    col_labels = plot_df.columns.astype(str).tolist()

    values = plot_df.values.astype(float)

    plt.figure(
        figsize=(
            max(7, 1.6 * len(col_labels)),
            max(5, 0.42 * len(row_labels))
        ),
        dpi=300
    )

    im = plt.imshow(values, aspect="auto")

    cbar = plt.colorbar(im)
    cbar.set_label(value_label)

    plt.xticks(
        np.arange(len(col_labels)),
        col_labels,
        rotation=35,
        ha="right",
        fontsize=10
    )

    plt.yticks(
        np.arange(len(row_labels)),
        row_labels,
        fontsize=9
    )

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            value = values[i, j]

            if not np.isnan(value):
                plt.text(
                    j,
                    i,
                    f"{value:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8
                )

    plt.title(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


def save_delta_bar(delta_df, feature_set_name, metric, out_png, title):
    """
    Plot the performance difference relative to Classical_only.
    """
    sub = delta_df[
        delta_df["Feature_set"] == feature_set_name
    ].copy()

    delta_col = f"Delta_{metric}_vs_Classical_only"

    if sub.empty or delta_col not in sub.columns:
        print("Skipped empty figure:", title)
        return

    sub["Label"] = (
        sub["Model"].astype(str)
        + " | "
        + sub["Imputation_method"].astype(str)
    )

    sub = sub.sort_values(delta_col, ascending=True)

    plt.figure(figsize=(9, max(5, 0.35 * len(sub))), dpi=300)

    plt.barh(sub["Label"], sub[delta_col])
    plt.axvline(0, linestyle="--", linewidth=1)

    plt.xlabel(
        f"Delta {metric} vs Classical_only",
        fontsize=12,
        fontweight="bold"
    )

    plt.ylabel(
        "Model | Imputation workflow",
        fontsize=12,
        fontweight="bold"
    )

    plt.title(title, fontsize=14, fontweight="bold")

    for i, value in enumerate(sub[delta_col].values):
        plt.text(value, i, f"{value:.3f}", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


def save_excel_with_autowidth(path, sheets):
    """
    Save multiple dataframes to Excel and adjust column widths.
    """
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet_name, df in sheets.items():
            if df is None or df.empty:
                continue

            safe_sheet = sheet_name[:31]

            df.to_excel(
                writer,
                sheet_name=safe_sheet,
                index=False
            )

            worksheet = writer.book[safe_sheet]

            for col_cells in worksheet.columns:
                max_len = 0
                col_letter = col_cells[0].column_letter

                for cell in col_cells:
                    try:
                        cell_value = str(cell.value) if cell.value is not None else ""
                        max_len = max(max_len, len(cell_value))
                    except Exception:
                        pass

                adjusted_width = min(max(max_len + 2, 10), 45)
                worksheet.column_dimensions[col_letter].width = adjusted_width

            worksheet.freeze_panes = "A2"


# ============================================================
# 3. Read seven-classifier result workbooks
# ============================================================

all_fold_metrics = []
all_summary_from_files = []
all_classwise_metrics = []
all_confusion_long = []
read_log = []

for model_name in MODEL_ORDER:
    model_dir = BASE_DIR / model_name

    if not model_dir.exists():
        read_log.append({
            "Model": model_name,
            "Status": "missing_model_dir",
            "Path": str(model_dir)
        })
        continue

    result_file = find_model_result_file(model_dir)

    if result_file is None:
        read_log.append({
            "Model": model_name,
            "Status": "missing_result_file",
            "Path": str(model_dir)
        })
        continue

    print(f"Reading {model_name}: {result_file}")

    xls = pd.ExcelFile(result_file)

    if "fold_metrics" in xls.sheet_names:
        df = pd.read_excel(result_file, sheet_name="fold_metrics")
        df = standardize_common_columns(df)
        df = to_numeric_metrics(df)

        if "Model" not in df.columns:
            df["Model"] = model_name

        all_fold_metrics.append(df)

    if "summary_mean_std" in xls.sheet_names:
        df = pd.read_excel(result_file, sheet_name="summary_mean_std")
        df = standardize_common_columns(df)

        if "Model" not in df.columns:
            df["Model"] = model_name

        all_summary_from_files.append(df)

    if "classwise_metrics" in xls.sheet_names:
        df = pd.read_excel(result_file, sheet_name="classwise_metrics")
        df = standardize_common_columns(df)

        if "Model" not in df.columns:
            df["Model"] = model_name

        all_classwise_metrics.append(df)

    if "confusion_matrix_long" in xls.sheet_names:
        df = pd.read_excel(result_file, sheet_name="confusion_matrix_long")
        df = standardize_common_columns(df)

        if "Model" not in df.columns:
            df["Model"] = model_name

        all_confusion_long.append(df)

    read_log.append({
        "Model": model_name,
        "Status": "ok",
        "Path": str(result_file),
        "Sheets": ", ".join(xls.sheet_names)
    })


read_log_df = pd.DataFrame(read_log)

if not all_fold_metrics:
    raise ValueError(
        "No fold_metrics sheets were loaded. Please check the classifier result files."
    )

fold_metrics_df = pd.concat(all_fold_metrics, axis=0, ignore_index=True)
fold_metrics_df = standardize_common_columns(fold_metrics_df)
fold_metrics_df = to_numeric_metrics(fold_metrics_df)
fold_metrics_df = sort_by_order(fold_metrics_df)

summary_from_files_df = (
    pd.concat(all_summary_from_files, axis=0, ignore_index=True)
    if all_summary_from_files else pd.DataFrame()
)

summary_from_files_df = standardize_common_columns(summary_from_files_df)

if not summary_from_files_df.empty:
    summary_from_files_df = sort_by_order(summary_from_files_df)

classwise_df = (
    pd.concat(all_classwise_metrics, axis=0, ignore_index=True)
    if all_classwise_metrics else pd.DataFrame()
)

classwise_df = standardize_common_columns(classwise_df)

if not classwise_df.empty:
    classwise_df = sort_by_order(classwise_df)

confusion_df = (
    pd.concat(all_confusion_long, axis=0, ignore_index=True)
    if all_confusion_long else pd.DataFrame()
)

confusion_df = standardize_common_columns(confusion_df)

if not confusion_df.empty:
    confusion_df = sort_by_order(confusion_df)

print("\nRead log:")
print(read_log_df.to_string(index=False))

print("\nTotal fold_metrics rows:", len(fold_metrics_df))
print(fold_metrics_df.head().to_string(index=False))


# ============================================================
# 4. Recalculate global summary tables
# ============================================================

summary_df = (
    fold_metrics_df
    .groupby(["Model", "Imputation_method", "Feature_set"], as_index=False)
    .agg({
        "N_features": ["mean", "std"],
        "Accuracy": ["mean", "std"],
        "Balanced_accuracy": ["mean", "std"],
        "Macro_precision": ["mean", "std"],
        "Macro_recall": ["mean", "std"],
        "Macro_F1": ["mean", "std"],
        "Weighted_precision": ["mean", "std"],
        "Weighted_recall": ["mean", "std"],
        "Weighted_F1": ["mean", "std"],
    })
)

summary_df = flatten_columns(summary_df)
summary_df = sort_by_order(summary_df)

overall_summary_df = (
    fold_metrics_df
    .groupby(["Model", "Feature_set"], as_index=False)
    .agg({
        "N_features": ["mean", "std"],
        "Accuracy": ["mean", "std"],
        "Balanced_accuracy": ["mean", "std"],
        "Macro_precision": ["mean", "std"],
        "Macro_recall": ["mean", "std"],
        "Macro_F1": ["mean", "std"],
        "Weighted_precision": ["mean", "std"],
        "Weighted_recall": ["mean", "std"],
        "Weighted_F1": ["mean", "std"],
    })
)

overall_summary_df = flatten_columns(overall_summary_df)
overall_summary_df = sort_by_order(overall_summary_df)


# ============================================================
# 5. Best feature set and delta relative to Classical_only
# ============================================================

best_by_model_method_df = (
    summary_df
    .sort_values(
        ["Model", "Imputation_method", "Macro_F1_mean"],
        ascending=[True, True, False]
    )
    .groupby(["Model", "Imputation_method"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_overall_by_model_df = (
    overall_summary_df
    .sort_values(
        ["Model", "Macro_F1_mean"],
        ascending=[True, False]
    )
    .groupby(["Model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

baseline = summary_df[
    summary_df["Feature_set"] == "Classical_only"
][[
    "Model",
    "Imputation_method",
    "Macro_F1_mean",
    "Balanced_accuracy_mean",
    "Accuracy_mean"
]].copy()

baseline = baseline.rename(columns={
    "Macro_F1_mean": "Classical_only_Macro_F1_mean",
    "Balanced_accuracy_mean": "Classical_only_Balanced_accuracy_mean",
    "Accuracy_mean": "Classical_only_Accuracy_mean"
})

delta_df = summary_df.merge(
    baseline,
    on=["Model", "Imputation_method"],
    how="left"
)

delta_df["Delta_Macro_F1_vs_Classical_only"] = (
    delta_df["Macro_F1_mean"] - delta_df["Classical_only_Macro_F1_mean"]
)

delta_df["Delta_Balanced_accuracy_vs_Classical_only"] = (
    delta_df["Balanced_accuracy_mean"]
    - delta_df["Classical_only_Balanced_accuracy_mean"]
)

delta_df["Delta_Accuracy_vs_Classical_only"] = (
    delta_df["Accuracy_mean"] - delta_df["Classical_only_Accuracy_mean"]
)

delta_df = sort_by_order(delta_df)

win_count_df = (
    delta_df[delta_df["Feature_set"] != "Classical_only"]
    .groupby("Feature_set", as_index=False)
    .agg(
        N_comparisons=("Delta_Macro_F1_vs_Classical_only", "count"),
        N_positive_delta_Macro_F1=(
            "Delta_Macro_F1_vs_Classical_only",
            lambda x: int((x > 0).sum())
        ),
        Mean_delta_Macro_F1=("Delta_Macro_F1_vs_Classical_only", "mean"),
        Median_delta_Macro_F1=("Delta_Macro_F1_vs_Classical_only", "median"),
        N_positive_delta_Balanced_accuracy=(
            "Delta_Balanced_accuracy_vs_Classical_only",
            lambda x: int((x > 0).sum())
        ),
        Mean_delta_Balanced_accuracy=("Delta_Balanced_accuracy_vs_Classical_only", "mean"),
        Median_delta_Balanced_accuracy=("Delta_Balanced_accuracy_vs_Classical_only", "median"),
    )
)

win_count_df["Positive_rate_Macro_F1"] = (
    win_count_df["N_positive_delta_Macro_F1"]
    / win_count_df["N_comparisons"]
)

win_count_df["Positive_rate_Balanced_accuracy"] = (
    win_count_df["N_positive_delta_Balanced_accuracy"]
    / win_count_df["N_comparisons"]
)


# ============================================================
# 6. Class-wise summary
# ============================================================

if not classwise_df.empty:
    for col in ["Precision", "Recall", "F1", "Support"]:
        if col in classwise_df.columns:
            classwise_df[col] = pd.to_numeric(classwise_df[col], errors="coerce")

    classwise_summary_df = (
        classwise_df
        .groupby(["Model", "Imputation_method", "Feature_set", "Class"], as_index=False)
        .agg({
            "Precision": ["mean", "std"],
            "Recall": ["mean", "std"],
            "F1": ["mean", "std"],
            "Support": ["mean", "std"],
        })
    )

    classwise_summary_df = flatten_columns(classwise_summary_df)
    classwise_summary_df = sort_by_order(classwise_summary_df)

    s_type_summary_df = classwise_summary_df[
        classwise_summary_df["Class"] == "S"
    ].copy()

else:
    classwise_summary_df = pd.DataFrame()
    s_type_summary_df = pd.DataFrame()


# ============================================================
# 7. Tables for reporting
# ============================================================

macro_f1_mean_std_table = make_mean_std_table(summary_df, "Macro_F1")
balanced_acc_mean_std_table = make_mean_std_table(summary_df, "Balanced_accuracy")
accuracy_mean_std_table = make_mean_std_table(summary_df, "Accuracy")

overall_macro_f1_table = make_overall_mean_std_table(
    overall_summary_df,
    "Macro_F1"
)

overall_balanced_acc_table = make_overall_mean_std_table(
    overall_summary_df,
    "Balanced_accuracy"
)

overall_accuracy_table = make_overall_mean_std_table(
    overall_summary_df,
    "Accuracy"
)

macro_f1_pivot_by_method = make_pivot_by_method(summary_df, "Macro_F1")
balanced_acc_pivot_by_method = make_pivot_by_method(summary_df, "Balanced_accuracy")
accuracy_pivot_by_method = make_pivot_by_method(summary_df, "Accuracy")

overall_macro_f1_pivot = make_overall_pivot(overall_summary_df, "Macro_F1")
overall_balanced_acc_pivot = make_overall_pivot(overall_summary_df, "Balanced_accuracy")
overall_accuracy_pivot = make_overall_pivot(overall_summary_df, "Accuracy")


# ============================================================
# 8. Two-panel heatmaps
# ============================================================

def make_metric_matrix_for_panel(summary_df, metric, method):
    """
    Build a matrix with feature sets as rows and classifiers as columns.
    """
    metric_col = f"{metric}_mean"

    sub = summary_df[
        summary_df["Imputation_method"].astype(str) == method
    ].copy()

    pivot = sub.pivot_table(
        index="Feature_set",
        columns="Model",
        values=metric_col,
        aggfunc="first"
    )

    pivot = pivot.reindex(index=FEATURE_SET_ORDER_PLOT)
    pivot = pivot.reindex(columns=MODEL_ORDER_PLOT)

    return pivot


def draw_one_heatmap_panel(
    ax,
    matrix,
    panel_title,
    vmin,
    vmax,
    show_ylabels=True
):
    """
    Draw one heatmap panel.
    """
    values = matrix.values.astype(float)

    im = ax.imshow(
        values,
        aspect="auto",
        vmin=vmin,
        vmax=vmax
    )

    x_labels = [
        MODEL_LABEL_MAP_PLOT.get(model, model)
        for model in matrix.columns.astype(str).tolist()
    ]

    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_xticklabels(
        x_labels,
        fontsize=12,
        fontweight="bold"
    )

    y_labels = [
        FEATURE_LABEL_MAP_PLOT.get(feature_set, feature_set)
        for feature_set in matrix.index.astype(str).tolist()
    ]

    ax.set_yticks(np.arange(len(y_labels)))

    if show_ylabels:
        ax.set_yticklabels(
            y_labels,
            fontsize=12,
            fontweight="bold"
        )
    else:
        ax.set_yticklabels([])

    ax.set_title(
        panel_title,
        fontsize=15,
        fontweight="bold",
        pad=10
    )

    ax.set_xlabel(
        "Classifier",
        fontsize=13,
        fontweight="bold",
        labelpad=8
    )

    ax.set_xticks(np.arange(-0.5, len(x_labels), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(y_labels), 1), minor=True)

    ax.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=1.5
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            value = values[i, j]

            if not np.isnan(value):
                ax.text(
                    j,
                    i,
                    f"{value:.3f}",
                    ha="center",
                    va="center",
                    fontsize=13,
                    fontweight="bold",
                    color="white"
                )

    for spine in ax.spines.values():
        spine.set_linewidth(1.4)

    return im


def plot_two_panel_heatmap(
    summary_df,
    metric,
    out_png,
    main_title,
    cbar_label,
    vmin=None,
    vmax=None
):
    """
    Plot two panels:
    (a) global-mean
    (b) KNN
    """
    matrix_global = make_metric_matrix_for_panel(
        summary_df,
        metric=metric,
        method="global_mean"
    )

    matrix_knn = make_metric_matrix_for_panel(
        summary_df,
        metric=metric,
        method="knn"
    )

    all_values = np.concatenate([
        matrix_global.values.astype(float).ravel(),
        matrix_knn.values.astype(float).ravel()
    ])

    all_values = all_values[~np.isnan(all_values)]

    if len(all_values) == 0:
        print(f"No valid values for {metric}. Heatmap skipped.")
        return

    if vmin is None:
        vmin = max(0.0, all_values.min() - 0.005)

    if vmax is None:
        vmax = min(1.0, all_values.max() + 0.005)

    fig = plt.figure(figsize=(15.5, 5.2), dpi=300)

    ax1 = fig.add_axes([0.08, 0.18, 0.39, 0.66])
    ax2 = fig.add_axes([0.54, 0.18, 0.39, 0.66])
    cax = fig.add_axes([0.955, 0.22, 0.018, 0.58])

    draw_one_heatmap_panel(
        ax=ax1,
        matrix=matrix_global,
        panel_title="(a) global-mean",
        vmin=vmin,
        vmax=vmax,
        show_ylabels=True
    )

    im2 = draw_one_heatmap_panel(
        ax=ax2,
        matrix=matrix_knn,
        panel_title="(b) KNN",
        vmin=vmin,
        vmax=vmax,
        show_ylabels=False
    )

    ax1.set_ylabel(
        "Feature set",
        fontsize=13,
        fontweight="bold",
        labelpad=8
    )

    cbar = fig.colorbar(
        im2,
        cax=cax
    )

    cbar.set_label(
        cbar_label,
        fontsize=12,
        fontweight="bold"
    )

    cbar.ax.tick_params(
        labelsize=10,
        width=1.2
    )

    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")

    fig.suptitle(
        main_title,
        fontsize=18,
        fontweight="bold",
        y=0.96
    )

    plt.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("Figure saved to:", out_png)


plot_two_panel_heatmap(
    summary_df=summary_df,
    metric="Macro_F1",
    out_png=OUT_DIR / "macroF1_heatmap_two_panels.png",
    main_title="Macro-F1 across classifiers and feature sets",
    cbar_label="Mean Macro-F1",
    vmin=0.78,
    vmax=0.93
)

plot_two_panel_heatmap(
    summary_df=summary_df,
    metric="Balanced_accuracy",
    out_png=OUT_DIR / "balanced_accuracy_heatmap_two_panels.png",
    main_title="Balanced accuracy across classifiers and feature sets",
    cbar_label="Mean balanced accuracy",
    vmin=0.78,
    vmax=0.93
)

plot_two_panel_heatmap(
    summary_df=summary_df,
    metric="Accuracy",
    out_png=OUT_DIR / "accuracy_heatmap_two_panels.png",
    main_title="Accuracy across classifiers and feature sets",
    cbar_label="Mean accuracy",
    vmin=0.82,
    vmax=0.94
)


# ============================================================
# 9. Delta and class-wise figures
# ============================================================

save_delta_bar(
    delta_df=delta_df,
    feature_set_name="Classical_plus_stable_novel",
    metric="Macro_F1",
    out_png=OUT_DIR / "delta_macroF1_classical_plus_stable_novel_vs_classical.png",
    title="Delta Macro-F1: Classical + stable novel vs Classical only"
)

save_delta_bar(
    delta_df=delta_df,
    feature_set_name="Stable_feature_set",
    metric="Macro_F1",
    out_png=OUT_DIR / "delta_macroF1_stable_feature_set_vs_classical.png",
    title="Delta Macro-F1: Stable feature set vs Classical only"
)

if not classwise_summary_df.empty:
    for feature_set_name in [
        "Classical_plus_stable_novel",
        "Stable_feature_set"
    ]:
        sub = classwise_summary_df[
            classwise_summary_df["Feature_set"] == feature_set_name
        ].copy()

        if not sub.empty:
            sub["Model_method"] = (
                sub["Model"].astype(str)
                + " | "
                + sub["Imputation_method"].astype(str)
            )

            class_f1_pivot = sub.pivot_table(
                index="Model_method",
                columns="Class",
                values="F1_mean",
                aggfunc="first"
            )

            class_f1_pivot = class_f1_pivot[
                [col for col in CLASS_ORDER if col in class_f1_pivot.columns]
            ]

            save_heatmap(
                class_f1_pivot,
                OUT_DIR / f"classwise_F1_heatmap_{feature_set_name}.png",
                f"Class-wise F1: {feature_set_name}",
                "Mean F1"
            )


# ============================================================
# 10. Save Excel summary
# ============================================================

sheets = {
    "read_log": read_log_df,
    "all_fold_metrics": fold_metrics_df,
    "summary_by_model_method": summary_df,
    "overall_by_model_feature": overall_summary_df,

    "overall_macroF1_table": overall_macro_f1_table,
    "overall_balAcc_table": overall_balanced_acc_table,
    "overall_accuracy_table": overall_accuracy_table,

    "best_by_model_method": best_by_model_method_df,
    "best_overall_by_model": best_overall_by_model_df,

    "delta_vs_classical": delta_df,
    "win_count_vs_classical": win_count_df,

    "macroF1_mean_std_table": macro_f1_mean_std_table,
    "balancedAcc_mean_std": balanced_acc_mean_std_table,
    "accuracy_mean_std": accuracy_mean_std_table,

    "pivot_macroF1_by_method": macro_f1_pivot_by_method.reset_index(),
    "pivot_balAcc_by_method": balanced_acc_pivot_by_method.reset_index(),
    "pivot_accuracy_by_method": accuracy_pivot_by_method.reset_index(),

    "pivot_overall_macroF1": overall_macro_f1_pivot.reset_index(),
    "pivot_overall_balAcc": overall_balanced_acc_pivot.reset_index(),
    "pivot_overall_accuracy": overall_accuracy_pivot.reset_index(),

    "classwise_all": classwise_df,
    "classwise_summary": classwise_summary_df,
    "S_type_summary": s_type_summary_df,
    "confusion_matrix_long": confusion_df,
}

save_excel_with_autowidth(OUT_XLSX, sheets)


# ============================================================
# 11. Print core results
# ============================================================

print("\n========== Seven-classifier summary completed ==========")
print("Output directory:", OUT_DIR)
print("Excel summary file:", OUT_XLSX)

print("\n========== Best feature set under each model-imputation combination ==========")

show_cols = [
    "Model",
    "Imputation_method",
    "Feature_set",
    "N_features_mean",
    "Accuracy_mean",
    "Balanced_accuracy_mean",
    "Macro_F1_mean",
    "Macro_F1_std",
]

show_cols = [
    col for col in show_cols
    if col in best_by_model_method_df.columns
]

print(
    best_by_model_method_df[show_cols]
    .sort_values(["Model", "Imputation_method"])
    .reset_index(drop=True)
    .to_string(index=False)
)

print("\n========== Win counts relative to Classical_only ==========")
print(win_count_df.to_string(index=False))

print("\n========== Overall Macro-F1 pivot table ==========")

if not overall_macro_f1_pivot.empty:
    print(overall_macro_f1_pivot.to_string())

print("\n========== Macro-F1 mean +/- SD table ==========")

if not macro_f1_mean_std_table.empty:
    print(macro_f1_mean_std_table.to_string(index=False))